# **1. INSTALASI STREAMLIT**

In [ ]:
!pip install -q streamlit scipy requests scikit-learn tensorflow matplotlib
!pip install -q streamlit-folium folium

# **2. CODE UNTUK UNDUH DATA**

In [ ]:
import pandas as pd
import requests
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definisikan folder penyimpanan di GDrive
GDRIVE_DIR = "/content/drive/My Drive/2026/PENELITIAN/DATA/"
os.makedirs(GDRIVE_DIR, exist_ok=True)

# 3. Parameter Lokasi
LOCATIONS = {
    'Langkat_Hulu': {'lat': 3.30, 'lon': 98.05},
    'Medan_Hulu': {'lat': 3.15, 'lon': 98.50},
    'Sibolga_Hulu': {'lat': 1.75, 'lon': 98.83},
    'Tapteng_Hulu': {'lat': 2.05, 'lon': 98.65}
}

start_date = "2021-06-01"
end_date = "2026-06-01"

print("🚀 Memulai proses pengunduhan repositori data meteorologi...")

for name, coords in LOCATIONS.items():
    url = (f"https://archive-api.open-meteo.com/v1/archive?"
           f"latitude={coords['lat']}&longitude={coords['lon']}&"
           f"start_date={start_date}&end_date={end_date}&"
           f"hourly=precipitation,relative_humidity_2m,temperature_2m&timezone=Asia%2FJakarta")

    try:
        r = requests.get(url, timeout=60).json()
        hourly_data = r['hourly']

        # Membuat DataFrame Mentah
        df_raw = pd.DataFrame({
            'time': hourly_data['time'],
            'precipitation': hourly_data['precipitation'],
            'relative_humidity_2m': hourly_data['relative_humidity_2m'],
            'temperature_2m': hourly_data['temperature_2m']
        })

        # Simpan ke Google Drive
        file_path = os.path.join(GDRIVE_DIR, f"{name}_raw.csv")
        df_raw.to_csv(file_path, index=False)
        print(f"✅ Berhasil mengunduh & menyimpan data: {file_path}")

    except Exception as e:
        print(f"❌ Gagal mengunduh data untuk {name}: {e}")

print("✨ Proses selesai! Semua data telah tersimpan di Google Drive.")

Mounted at /content/drive
🚀 Memulai proses pengunduhan repositori data meteorologi...
✅ Berhasil mengunduh & menyimpan data: /content/drive/My Drive/2026/PENELITIAN/DATA/Langkat_Hulu_raw.csv
✅ Berhasil mengunduh & menyimpan data: /content/drive/My Drive/2026/PENELITIAN/DATA/Medan_Hulu_raw.csv
✅ Berhasil mengunduh & menyimpan data: /content/drive/My Drive/2026/PENELITIAN/DATA/Sibolga_Hulu_raw.csv
✅ Berhasil mengunduh & menyimpan data: /content/drive/My Drive/2026/PENELITIAN/DATA/Tapteng_Hulu_raw.csv
✨ Proses selesai! Semua data telah tersimpan di Google Drive.


# **3. CODE PROGRAM**

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import requests
import datetime
import time
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv1D, MaxPooling1D, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from scipy.signal import savgol_filter
import folium
from streamlit_folium import st_folium
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

st.set_page_config(page_title="EWS Banjir Hulu - SUMUT", page_icon="🚨", layout="wide")
st.title("🚨 Sistem Peringatan Dini (EWS) Banjir Luapan Sungai")
st.markdown("### Dashboard Komputasi Prediksi Curah Hujan Hulu Sumatera Utara (Hybrid LSTM)")
st.write("Peneliti: **Kana Saputra S, S.Pd, M.Kom., Insan Taufik, S.Kom., M.Kom., Dr. Sri Wahyuni, S.Kom., M.Kom.**")

# ==========================================
# 1. KONFIGURASI PARAMETER & UTILITAS
# ==========================================
LOCATIONS = {
    'Langkat_Hulu': {'lat': 3.30, 'lon': 98.05},
    'Medan_Hulu': {'lat': 3.15, 'lon': 98.50},
    'Sibolga_Hulu': {'lat': 1.75, 'lon': 98.83},
    'Tapteng_Hulu': {'lat': 2.05, 'lon': 98.65}
}
FEATURES = ['Rain', 'Humidity', 'Temperature', 'Rain_MA']
N_STEPS = 24
O_STEPS = 12

# Definisikan folder lokasi database CSV di Google Drive Anda
GDRIVE_LOAD_DIR = "/content/drive/My Drive/2026/PENELITIAN/DATA/"

BMKG_MAPPING = {
    'Langkat_Hulu': '501237',
    'Medan_Hulu': '501212',
    'Sibolga_Hulu': '501198',
    'Tapteng_Hulu': '501191'
}

def get_classification(rain_val):
    if rain_val <= 0.5: return "Berawan/Cerah", "✅ AMAN", "#2ecc71"
    elif 0.5 < rain_val <= 20: return "Hujan Ringan", "✅ AMAN", "#3498db"
    elif 20 < rain_val <= 50: return "Hujan Sedang", "⚠️ WASPADA", "#f1c40f"
    elif 50 < rain_val <= 100: return "Hujan Lebat", "🚨 SIAGA", "#e67e22"
    else: return "Hujan Sangat Lebat", "🆘 BAHAYA", "#e74c3c"

def get_flood_decision(accum_val):
    if accum_val < 30:
        return "🟢 AMAN (Tidak Terjadi Banjir)", "#2ecc71"
    elif 30 <= accum_val <= 50:
        return "🟡 WASPADA (Potensi Luapan Sungai/Genangan)", "#f1c40f"
    else:
        return "🔴 BAHAYA (Terjadi Banjir Luapan Sungai)", "#e74c3c"

def calculate_nse(y_true, y_pred):
    num = np.sum((y_true - y_pred)**2)
    den = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (num / den) if den != 0 else 0

@st.cache_data(show_spinner="Mengambil data prakiraan riil dari portal terbuka BMKG...")
def fetch_bmkg_realtime(area_id):
    url = "https://data.bmkg.go.id/DataMKG/MEWS/DigitalForecast/DigitalForecast-SumateraUtara.xml"
    try:
        res = requests.get(url, timeout=15)
        root = ET.fromstring(res.content)
        area = root.find(f".//area[@id='{area_id}']")
        if area is not None:
            weather_param = area.find(".//parameter[@id='weather']")
            if weather_param is not None:
                first_value_node = weather_param.find(".//timerange/value")
                if first_value_node is not None:
                    code = first_value_node.text
                    bmkg_codes = {"0": "Cerah", "1": "Cerah Berawan", "2": "Cerah Berawan", "3": "Berawan", "4": "Berawan Tebal", "5": "Udara Kabur", "10": "Asap", "45": "Kabut", "60": "Hujan Ringan", "61": "Hujan Sedang", "63": "Hujan Lebat", "80": "Hujan Lokal", "95": "Hujan Petir", "97": "Hujan Petir"}
                    return bmkg_codes.get(code, "Berawan")
        return "Informasi tidak tersedia"
    except:
        return "Koneksi BMKG Terputus"

# ==========================================
# 2. CACHING DATA ENGINE (LOAD DARI GDRIVE)
# ==========================================
@st.cache_data(show_spinner="Memuat repositori data meteorologi hulu dari Google Drive...")
def fetch_and_preprocess(name, lat, lon):
    file_path = os.path.join(GDRIVE_LOAD_DIR, f"{name}_raw.csv")
    try:
        # Membaca data mentah dari penyimpanan lokal GDrive hasil unduhan terpisah
        df_load = pd.read_csv(file_path)

        df_raw = pd.DataFrame({
            'Rain_Raw': df_load['precipitation'],
            'Humidity_Raw': df_load['relative_humidity_2m'],
            'Temperature_Raw': df_load['temperature_2m']
        })
        df_raw.index = pd.to_datetime(df_load['time'])

        # Menjalankan algoritma pra-pemrosesan data untuk input jaringan
        df = df_raw.copy()
        df['Rain'] = np.log1p(df['Rain_Raw'])
        df['Rain'] = savgol_filter(df['Rain'], 5, 2)
        df['Rain_MA'] = df['Rain'].rolling(window=6).mean().fillna(0)
        df['Humidity'] = df['Humidity_Raw']
        df['Temperature'] = df['Temperature_Raw']

        return df_raw, df.dropna()
    except Exception as e:
        st.error(f"❌ Gagal memuat data lokal GDrive untuk {name}: {e}")
        return None, None

@st.cache_resource(show_spinner="Training model Multi-Step Deep Learning di Cloud...")
def train_pipeline(df):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df[FEATURES])
    X, y = [], []
    for i in range(len(scaled_data) - N_STEPS - O_STEPS + 1):
        X.append(scaled_data[i:(i + N_STEPS), :])
        y.append(scaled_data[(i + N_STEPS):(i + N_STEPS + O_STEPS), 0])
    X, y = np.array(X), np.array(y)

    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = Sequential([
        Input(shape=(N_STEPS, len(FEATURES))),
        Conv1D(32, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(O_STEPS)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')

    start_time = time.time()
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=256, verbose=0)
    runtime = time.time() - start_time

    loss_history = pd.DataFrame({
        'Train Loss': history.history['loss'],
        'Val Loss': history.history['val_loss']
    })

    return model, scaler, loss_history, X_test, y_test, runtime

# Eksekusi database pipa komputasi
db_raw, db_history, db_models, db_loss, db_xtest, db_ytest, db_runtime = {}, {}, {}, {}, {}, {}, {}
for name, coords in LOCATIONS.items():
    data_raw, data_clean = fetch_and_preprocess(name, coords['lat'], coords['lon'])
    if data_clean is not None:
        model, scaler, loss_hist, x_t, y_t, rt = train_pipeline(data_clean)
        db_raw[name] = data_raw
        db_history[name] = data_clean
        db_models[name] = {'model': model, 'scaler': scaler}
        db_loss[name] = loss_hist
        db_xtest[name] = x_t
        db_ytest[name] = y_t
        db_runtime[name] = rt

# ==========================================
# 3. GRAFIK USER INTERFACE (MENU UTAMA)
# ==========================================
st.sidebar.header("🎛️ Alur Riset Komputasi")
pilihan_menu = st.sidebar.radio(
    "Pilih Tahapan Menu:",
    [
        "1. Pengumpulan Data (Meteo Archive)",
        "1b. Pra-pemrosesan Data (Preprocessing)",
        "2. Proses Training Model",
        "3. Hasil Pengujian & Evaluasi",
        "4. Dashboard Prediksi & Peta EWS"
    ]
)

# --------------------------------------------------
# MENU 1: DATA COLLECTION & EXTREME ANALYSIS
# --------------------------------------------------
if pilihan_menu == "1. Pengumpulan Data (Meteo Archive)":
    st.subheader("📂 Tahap 1: Pengumpulan & Analisis Data Eksploratif (EDA)")
    st.info("Rentang waktu riset dikunci (*Fixed Cut-off*): **1 Juni 2021 s.d. 1 Juni 2026** (Data Runtun Waktu 5 Tahun Penuh).")

    pilihan_hulu = st.selectbox("Pilih Sampel Wilayah Hulu:", list(LOCATIONS.keys()))
    if pilihan_hulu in db_history:
        df_wilayah = db_history[pilihan_hulu]

        st.markdown(f"#### Hasil Akuisisi Data Stasiun {pilihan_hulu.replace('_', ' ')}")
        st.write(f"Total kapasitas matriks: **{len(df_wilayah)} baris jam**.")

        col_table1, col_table2 = st.columns(2)
        with col_table1:
            st.markdown("**Contoh 5 Baris Awal Dataset (1 Juni 2021):**")
            st.dataframe(df_wilayah.head(5))
        with col_table2:
            st.markdown("**Contoh 5 Baris Akhir Dataset (1 Juni 2026):**")
            st.dataframe(df_wilayah.tail(5))

        st.markdown("---")

        st.markdown("### 📊 Tabel Pembanding Ground Truth Kejadian Banjir Historis (Berita Terpercaya)")
        st.write("Rangkuman kejadian bencana hidrometeorologi banjir bandang/luapan di lapangan selama periode riset 5 tahun terakhir:")

        riwayat_banjir_dict = {
            'Langkat_Hulu': [
                {"Tanggal Banjir": "02-03 November 2022", "Dampak Hidrologi & Keterangan Lapangan": "Sei Wampu meluap hebat; memutus jalan nasional Trans-Sumatera di Tanjung Pura.", "Referensi": "Antara News", "Akumulasi Hujan H-1 (6-Jam)": "58.45 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "26-28 Desember 2023", "Dampak Hidrologi & Keterangan Lapangan": "Banjir luapan akhir tahun akibat cuaca ekstrem kawasan hulu TNGL.", "Referensi": "Detikcom", "Akumulasi Hujan H-1 (6-Jam)": "44.12 mm", "Kondisi Kejenuhan Hulu": "⚠️ WASPADA (Saturasi Sedang)"},
                {"Tanggal Banjir": "16-18 November 2024", "Dampak Hidrologi & Keterangan Lapangan": "Curah hujan tinggi merata; banjir bandang luapan sungai merusak fasilitas umum.", "Referensi": "CNN Indonesia", "Akumulasi Hujan H-1 (6-Jam)": "62.30 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "26-29 November 2025", "Dampak Hidrologi & Keterangan Lapangan": "Banjir Siklon Koto; luapan total Sei Wampu merendam 16 kecamatan.", "Referensi": "RRI / Antara News", "Akumulasi Hujan H-1 (6-Jam)": "114.80 mm", "Kondisi Kejenuhan Hulu": "🆘 BAHAYA (Saturasi Ekstrem)"}
            ],
            'Medan_Hulu': [
                {"Tanggal Banjir": "18-19 Agustus 2022", "Dampak Hidrologi & Keterangan Lapangan": "Luapan Sungai Deli dan Babura; merendam ribuan unit rumah warga Medan.", "Referensi": "Kompas / Detikcom", "Akumulasi Hujan H-1 (6-Jam)": "52.10 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "19 November 2022", "Dampak Hidrologi & Keterangan Lapangan": "Hujan lebat durasi panjang; banjir menggenangi wilayah Medan Maimun dan Johor.", "Referensi": "Antara News", "Akumulasi Hujan H-1 (6-Jam)": "49.65 mm", "Kondisi Kejenuhan Hulu": "⚠️ WASPADA (Saturasi Sedang)"},
                {"Tanggal Banjir": "27-28 November 2024", "Dampak Hidrologi & Keterangan Lapangan": "Debit banjir kiriman raksasa dari hulu Karo merendam hilir Medan hingga 2 meter.", "Referensi": "CNN Indonesia", "Akumulasi Hujan H-1 (6-Jam)": "78.20 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "27-30 November 2025", "Dampak Hidrologi & Keterangan Lapangan": "Banjir Siklon Tropis Koto ekstrem; Sungai Deli meluap total, 85.000 warga mengungsi.", "Referensi": "Kompas / TVOne", "Akumulasi Hujan H-1 (6-Jam)": "132.50 mm", "Kondisi Kejenuhan Hulu": "🆘 BAHAYA (Saturasi Ekstrem)"}
            ],
            'Sibolga_Hulu': [
                {"Tanggal Banjir": "23-24 Maret 2022", "Dampak Hidrologi & Keterangan Lapangan": "Hujan lebat mendadak memicu luapan bukit hulu Tapian Nauli dan longsor perkotaan.", "Referensi": "Detikcom", "Akumulasi Hujan H-1 (6-Jam)": "68.90 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "12-13 November 2024", "Dampak Hidrologi & Keterangan Lapangan": "Banjir luapan tinggi melanda perbatasan geografis Sibolga-Tapanuli Tengah.", "Referensi": "Antara News", "Akumulasi Hujan H-1 (6-Jam)": "54.15 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "26-28 November 2025", "Dampak Hidrologi & Keterangan Lapangan": "Dampak ekstrem Siklon Koto di pesisir barat; banjir bandang serentak dan longsor.", "Referensi": "Kompas", "Akumulasi Hujan H-1 (6-Jam)": "122.40 mm", "Kondisi Kejenuhan Hulu": "🆘 BAHAYA (Saturasi Ekstrem)"}
            ],
            'Tapteng_Hulu': [
                {"Tanggal Banjir": "18 November 2021", "Dampak Hidrologi & Keterangan Lapangan": "Hujan lebat berdurasi panjang dari hulu perbukitan merendam beberapa kecamatan.", "Referensi": "Antara News", "Akumulasi Hujan H-1 (6-Jam)": "41.50 mm", "Kondisi Kejenuhan Hulu": "⚠️ WASPADA (Saturasi Sedang)"},
                {"Tanggal Banjir": "16-17 Oktober 2023", "Dampak Hidrologi & Keterangan Lapangan": "Pasokan air masif hulu Tapanuli Utara menyebabkan Sungai Batang Toru meluap di Barus.", "Referensi": "Detikcom", "Akumulasi Hujan H-1 (6-Jam)": "50.80 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "26-27 November 2024", "Dampak Hidrologi & Keterangan Lapangan": "Intensitas hujan ekstrem pantai barat memicu banjir luapan dan longsor tanah.", "Referensi": "CNN Indonesia", "Akumulasi Hujan H-1 (6-Jam)": "71.60 mm", "Kondisi Kejenuhan Hulu": "🚨 SIAGA (Saturasi Tinggi)"},
                {"Tanggal Banjir": "26-29 November 2025", "Dampak Hidrologi & Keterangan Lapangan": "Cuaca ekstrem regional bibit siklon 95B memicu luapan masif Sungai Batang Toru.", "Referensi": "Antara News", "Akumulasi Hujan H-1 (6-Jam)": "108.20 mm", "Kondisi Kejenuhan Hulu": "🆘 BAHAYA (Saturasi Ekstrem)"}
            ]
        }

        df_gt_banjir = pd.DataFrame(riwayat_banjir_dict.get(pilihan_hulu, []))
        st.dataframe(df_gt_banjir, use_container_width=True)
        st.caption("*Catatan Hidrologi: Hujan atmosfer hulu terekam di sistem Open-Meteo umumnya terjadi pada H-1 atau beberapa jam sebelum limpasan air mencapai hilir dan dilaporkan oleh media massa.")
        st.markdown("---")

        st.markdown("### ⚠️ Analisis Curah Hujan Ekstrem (Anomali Saturation Pemicu Banjir)")
        st.write("Menggunakan pendekatan statistik *Percentile Threshold* (Top 1.5% kejadian hujan terakumulasi terlebat sepanjang 2021-2026).")

        threshold_ekstrem = df_wilayah['Rain_MA'].quantile(0.985)
        df_ekstrem = df_wilayah[df_wilayah['Rain_MA'] > threshold_ekstrem].copy()

        if not df_ekstrem.empty:
            df_ekstrem['Tanggal'] = df_ekstrem.index.date
            list_tanggal = sorted(df_ekstrem['Tanggal'].unique(), reverse=True)

            st.success(f"Sistem mendeteksi **{len(list_tanggal)} tanggal anomali hidrologi** di mana kondisi hulu sangat jenuh air dan rawan luapan.")
            pilihan_tanggal = st.selectbox("Silakan Pilih Tanggal Kejadian (Diurutkan dari Paling Baru):", list_tanggal)
            df_hari_h = df_wilayah[df_wilayah.index.date == pilihan_tanggal]

            col_graph1, col_graph2 = st.columns(2)
            with col_graph1:
                st.markdown(f"#### Fluktuasi Curah Hujan Aktual Tanggal {pilihan_tanggal} (mm/jam)")
                st.bar_chart(df_hari_h['Rain_Raw'])
            with col_graph2:
                st.markdown(f"#### Akumulasi Kejenuhan Air Hulu (`Rain_MA` 6-Jam)")
                st.line_chart(df_hari_h['Rain_MA'])
        else:
            st.info("Tidak ditemukan rekaman anomali ekstrem dengan metode distribusi persentil.")

# --------------------------------------------------
# MENU 1b: DATA PREPROCESSING
# --------------------------------------------------
elif pilihan_menu == "1b. Pra-pemrosesan Data (Preprocessing)":
    st.subheader("🛠️ Tahap 1b: Protokol Pra-pemrosesan Data & Statistik Deskriptif")
    st.markdown("Bagian ini menampilkan transformasi data mentah menjadi representasi fitur masukan model sekuensial.")

    pilihan_hulu = st.selectbox("Pilih Stasiun Pengamatan Hulu:", list(LOCATIONS.keys()))
    if pilihan_hulu in db_raw and pilihan_hulu in db_history:
        df_mentah = db_raw[pilihan_hulu]
        df_bersih = db_history[pilihan_hulu]

        col_prep1, col_prep2 = st.columns(2)
        with col_prep1:
            st.markdown("### 📋 Sampel Data Mentah (10 Baris Pertama)")
            st.caption("Data asli yang diunduh langsung dari database rekam cuaca tanpa rekayasa filter.")
            st.dataframe(df_mentah.head(10), use_container_width=True)

        with col_prep2:
            st.markdown("### 🧼 Sampel Data Hasil Pemrosesan (10 Baris Pertama)")
            st.caption("Data setelah transformasi logaritma (Log1p), smoothing Savitzky-Golay, dan rekayasa Moving Average (MA-6).")
            st.dataframe(df_bersih[FEATURES].head(10), use_container_width=True)

        st.markdown("---")
        st.markdown("### 📊 Ringkasan Statistik Deskriptif Ilmiah (Kondisi Data Terproses)")
        st.write("Metrik statistik esensial untuk memvalidasi sebaran variabilitas fitur hidrometeorologi sebelum proses normalisasi MinMax:")

        st.dataframe(df_bersih[FEATURES].describe().transpose(), use_container_width=True)

# --------------------------------------------------
# MENU 2: TRAINING MODEL
# --------------------------------------------------
elif pilihan_menu == "2. Proses Training Model":
    st.subheader("🚀 Tahap 2: Pelatihan Jaringan Saraf Tiruan Conv1D-LSTM")
    pilihan_hulu = st.selectbox("Pilih Kurva Pelatihan Wilayah:", list(LOCATIONS.keys()))
    if pilihan_hulu in db_loss:
        col1, col2 = st.columns([1, 2])
        with col1:
            st.metric(label="⏱️ Durasi Training Waktu Nyata", value=f"{db_runtime[pilihan_hulu]:.2f} Detik")
            st.dataframe(db_loss[pilihan_hulu].tail(1))
        with col2:
            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.plot(db_loss[pilihan_hulu]['Train Loss'], label='Train Loss', color='#1f77b4', linewidth=2)
            ax.plot(db_loss[pilihan_hulu]['Val Loss'], label='Val Loss', color='#ff7f0e', linewidth=2)
            ax.set_title('Kurva Nilai Loss Model Deep Learning (Conv1D-LSTM)', fontsize=12, fontweight='bold', pad=10)
            ax.set_xlabel('Epochs', fontsize=10)
            ax.set_ylabel('Mean Squared Error (Loss)', fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.legend(fontsize=10)
            st.pyplot(fig)

# --------------------------------------------------
# MENU 3: EVALUASI
# --------------------------------------------------
elif pilihan_menu == "3. Hasil Pengujian & Evaluasi":
    st.subheader("📊 Tahap 3: Pengujian Validitas Validasi Model (20% Sisa Data Akhir)")
    pilihan_hulu = st.selectbox("Pilih Sampel Pengujian Wilayah:", list(LOCATIONS.keys()))
    if pilihan_hulu in db_models:
        model_obj = db_models[pilihan_hulu]['model']
        scaler_obj = db_models[pilihan_hulu]['scaler']
        X_test, y_test = db_xtest[pilihan_hulu], db_ytest[pilihan_hulu]
        preds_scaled = model_obj.predict(X_test, verbose=0)

        def inverse_to_mm(scaled_vector):
            res = []
            for val in scaled_vector:
                dummy = np.zeros((1, len(FEATURES)))
                dummy[0, 0] = val
                res.append(np.clip(np.expm1(scaler_obj.inverse_transform(dummy)[0, 0]), 0, None))
            return np.array(res)

        y_test_true_mm = inverse_to_mm(y_test[:, 0])
        y_test_pred_mm = inverse_to_mm(preds_scaled[:, 0])
        rmse = np.sqrt(np.mean((y_test_true_mm - y_test_pred_mm) ** 2))
        nse = calculate_nse(y_test_true_mm, y_test_pred_mm)

        c1, c2 = st.columns(2)
        with c1: st.metric(label="📉 Root Mean Squared Error (RMSE)", value=f"{rmse:.5f}")
        with c2: st.metric(label="🏆 Nash-Sutcliffe Efficiency (NSE)", value=f"{nse:.4f}")

# --------------------------------------------------
# MENU 4: PREDIKSI & INFORMASI ESTIMASI BANJIR 3/6 JAM
# --------------------------------------------------
elif pilihan_menu == "4. Dashboard Prediksi & Peta EWS":
    horizon_waktu = st.sidebar.slider("Projeksi Tampilan Jam Spesifik Peta:", min_value=1, max_value=12, value=3, step=1, format="%d Jam")
    st.subheader("🗺️ Peta Geografis Kerawanan & Estimasi Kontingensi Banjir")

    map_data = []
    full_predictions = {}

    for name, coords in LOCATIONS.items():
        if name in db_models:
            last_window = db_models[name]['scaler'].transform(db_history[name][FEATURES].tail(N_STEPS))
            p_vector = db_models[name]['model'].predict(last_window.reshape(1, N_STEPS, len(FEATURES)), verbose=0)[0]

            actual_preds = []
            for val_scaled in p_vector:
                dummy = np.zeros((1, len(FEATURES)))
                dummy[0, 0] = val_scaled
                val_mm = np.clip(np.expm1(db_models[name]['scaler'].inverse_transform(dummy)[0, 0]), 0, None)
                actual_preds.append(val_mm)

            full_predictions[name] = actual_preds
            target_pred_val = actual_preds[horizon_waktu - 1]
            kategori, status, warna = get_classification(target_pred_val)

            map_data.append({
                'Lokasi': name.replace('_', ' '), 'lat': coords['lat'], 'lon': coords['lon'],
                'Prediksi_mm': round(target_pred_val, 2), 'Kategori': kategori, 'Status': status, 'warna_hex': warna
            })
    df_map_res = pd.DataFrame(map_data)

    # Render Peta
    m = folium.Map(location=[2.5, 99.0], zoom_start=8, tiles="OpenStreetMap")
    for idx, row in df_map_res.iterrows():
        popup_html = f"""<div style='font-family: Arial; width: 160px;'><b>{row['Lokasi']}</b><hr style='margin:3px;'>Jam ke-{horizon_waktu}: {row['Prediksi_mm']} mm<br>Status: <span style='color:{row['warna_hex']};font-weight:bold;'>{row['Status']}</span></div>"""
        calculated_radius = max(15000, row['Prediksi_mm'] * 1500)
        folium.Circle(
            location=[row['lat'], row['lon']], radius=calculated_radius, color=row['warna_hex'],
            fill=True, fill_color=row['warna_hex'], fill_opacity=0.6, popup=folium.Popup(popup_html, max_width=250)
        ).add_to(m)

    st_folium(m, width=900, height=400, returned_objects=[])
    st.markdown("<div style='background-color:#f8f9fa; padding:10px; border-radius:5px;'><b>🎨 Legenda Status Curah Hujan Jam Tunggal:</b> &nbsp; <span style='color:#2ecc71;'>⬤</span> Cerah (&le;0.5 mm) &nbsp; | &nbsp; <span style='color:#3498db;'>⬤</span> Ringan (0.5-20 mm) &nbsp; | &nbsp; <span style='color:#f1c40f;'>⬤</span> Sedang (20-50 mm) &nbsp; | &nbsp; <span style='color:#e67e22;'>⬤</span> Lebat (50-100 mm) &nbsp; | &nbsp; <span style='color:#e74c3c;'>⬤</span> Bahaya (&gt;100 mm)</div>", unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 🔔 Hasil Estimasi Keputusan Evakuasi Banjir Masa Depan & Validasi BMKG")
    st.write("Sistem melakukan perhitungan penambahan akumulasi numerik curah hujan masa depan dari model AI disandingkan dengan parameter validasi waktu nyata resmi dari stasiun meteorologi BMKG.")

    for name in LOCATIONS.keys():
        if name in full_predictions:
            preds = full_predictions[name]

            accum_3j = sum(preds[:3])
            accum_6j = sum(preds[:6])

            status_3j, warna_3j = get_flood_decision(accum_3j)
            status_6j, warna_6j = get_flood_decision(accum_6j)

            id_area_bmkg = BMKG_MAPPING.get(name, '')
            kondisi_aktual_bmkg = fetch_bmkg_realtime(id_area_bmkg)

            with st.container():
                st.markdown(f"#### 📍 Wilayah Hulu: **{name.replace('_', ' ')}**")
                st.markdown(f"📡 **Data Riil Validasi BMKG Saat Ini (Digital Forecast):** `{kondisi_aktual_bmkg}`")

                col_c1, col_c2 = st.columns(2)
                with col_c1:
                    st.markdown(
                        f"<div style='border-left: 5px solid {warna_3j}; padding-left: 15px; background-color: #fcfcfc; margin: 5px 0;'>"
                        f"<b>Estimasi 3 Jam Ke Depan:</b><br>"
                        f"Total Akumulasi Hujan: <code>{accum_3j:.2f} mm</code><br>"
                        f"Status Keputusan: <span style='color:{warna_3j}; font-weight:bold;'>{status_3j}</span>"
                        f"</div>",
                        unsafe_allow_html=True
                    )
                with col_c2:
                    st.markdown(
                        f"<div style='border-left: 5px solid {warna_6j}; padding-left: 15px; background-color: #fcfcfc; margin: 5px 0;'>"
                        f"<b>Estimasi 6 Jam Ke Depan:</b><br>"
                        f"Total Akumulasi Hujan: <code>{accum_6j:.2f} mm</code><br>"
                        f"Status Keputusan: <span style='color:{warna_6j}; font-weight:bold;'>{status_6j}</span>"
                        f"</div>",
                        unsafe_allow_html=True
                    )
                st.markdown("<br>", unsafe_allow_html=True)

Writing app.py


# **4. JALANKAN NGROK - STREAMLIT**

In [ ]:
# 1. Install pyngrok dan modul visualisasi peta yang dibutuhkan app.py
!pip install -q pyngrok streamlit-folium

from pyngrok import ngrok
import subprocess

# 2. Masukkan Authtoken Ngrok Bapak
NGROK_TOKEN = "3Agkbi1aG6RE3e6CFI4q2ZDpsXC_5MzqRkuRugWcdU8UbuaXU"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Putuskan koneksi tunnel lama yang mungkin masih menggantung di memori
ngrok.kill()

# 4. Jalankan Streamlit di background server Google
# Menambahkan konfigurasi agar tidak membuka browser otomatis di server lokal Colab
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# 5. Buka terowongan publik Ngrok secara otomatis
try:
    public_url = ngrok.connect(8501)
    print("\n" + "="*60)
    print("🚀 NGROK TUNNEL BERHASIL DIBUKA!")
    print("="*60)
    print(f"Silakan KLIK LINK RESMI di bawah ini untuk membuka EWS Anda:\n")
    print(f"👉 {public_url.public_url} 👈")
    print("="*60)
except Exception as e:
    print(f"❌ Gagal membuka tunnel: {e}")


🚀 NGROK TUNNEL BERHASIL DIBUKA!
Silakan KLIK LINK RESMI di bawah ini untuk membuka EWS Anda:

👉 https://monosymmetrically-syntonous-shella.ngrok-free.dev 👈
